# Notebook 04: Classification Models

### Mục tiêu
- Train ≥2 baselines + mô hình cải tiến
- So sánh PR-AUC/F1, confusion matrix, feature importance
- Log thời gian train cho mỗi model

### Lý do chọn model
- **Logistic Regression** (baseline 1): đơn giản, interpretable, baseline chuẩn
- **Decision Tree** (baseline 2): dễ giải thích, capture non-linear patterns
- **Random Forest** (cải tiến 1): ensemble giảm overfitting, robust
- **XGBoost** (cải tiến 2): state-of-the-art gradient boosting, xử lý tốt imbalance

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from src.data.loader import load_config
from src.data.cleaner import encode_categoricals
from src.models.supervised import (
    split_data, train_and_evaluate,
    get_feature_importance, save_model,
)
from src.evaluation.metrics import (
    get_confusion_matrix, get_classification_report,
)
from src.evaluation.report import save_results
from src.visualization.plots import (
    plot_confusion_matrix, plot_multi_roc_pr,
    plot_feature_importance,
)

config = load_config("../configs/params.yaml")
seed = config['seed']

## 1. Chuẩn bị dữ liệu

In [2]:
df = pd.read_csv("../" + config["data"]["processed_path"])

# Drop non-numeric columns for modeling
drop_cols = ['arrival_date', 'lead_time_bin', 'season']
for col in drop_cols:
    if col in df.columns:
        df = df.drop(columns=[col])

# Encode categoricals
df = encode_categoricals(df)
print(f"Modeling data shape: {df.shape}")

[Cleaner] One-hot encoded: ['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']. New shape: (86966, 79)
Modeling data shape: (86966, 79)


## 2. Train/Test Split

In [3]:
X_train, X_test, y_train, y_test = split_data(
    df, test_size=config['test_size'], seed=seed,
)

[Supervised] Split: train=69,572, test=17,394 (positive rate: train=0.273, test=0.273)


## 3. Train và đánh giá models

Train 4 models: LogisticRegression, DecisionTree, RandomForest, XGBoost.
Mỗi model ghi nhận **thời gian train** (yêu cầu đề bài).

In [4]:
trained_models, results = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    seed=seed, cv_folds=config['classification']['cv_folds'],
)

# Save results table
save_results(results, "../outputs/tables/classification")
results


--- Training LogisticRegression ---
  Accuracy=0.7272, F1=0.6042, PR-AUC=0.6200, ROC-AUC=0.8121, Train time=8.68s

--- Training DecisionTree ---
  Accuracy=0.7030, F1=0.6136, PR-AUC=0.5239, ROC-AUC=0.8051, Train time=0.27s

--- Training RandomForest ---
  Accuracy=0.7723, F1=0.6732, PR-AUC=0.7404, ROC-AUC=0.8847, Train time=1.46s

--- Training XGBoost ---
  Accuracy=0.8526, F1=0.7156, PR-AUC=0.8088, ROC-AUC=0.9171, Train time=0.71s

[Supervised] === Model Comparison ===
             model  accuracy  precision   recall       f1  train_time_seconds  roc_auc   pr_auc
           XGBoost  0.852593   0.756212 0.679158 0.715617                0.71 0.917082 0.808788
      RandomForest  0.772278   0.553535 0.858737 0.673158                1.46 0.884706 0.740385
LogisticRegression  0.727205   0.500345 0.762316 0.604155                8.68 0.812082 0.619995
      DecisionTree  0.703001   0.475870 0.863579 0.613613                0.27 0.805084 0.523930
[Report] Saved CSV: ..\outputs\tables\classi

,model,accuracy,precision,recall,f1,train_time_seconds,roc_auc,pr_auc
0,XGBoost,0.852593,0.756212,0.679158,0.715617,0.71,0.917082,0.808788
1,RandomForest,0.772278,0.553535,0.858737,0.673158,1.46,0.884706,0.740385
2,LogisticRegression,0.727205,0.500345,0.762316,0.604155,8.68,0.812082,0.619995
3,DecisionTree,0.703001,0.475870,0.863579,0.613613,0.27,0.805084,0.523930


## 4. Confusion Matrix (best model)

In [5]:
best_name = results.iloc[0]['model']
print(f"Best model: {best_name}")

best_data = trained_models[best_name]
plot_confusion_matrix(y_test, best_data["y_pred"],
    title=f"Confusion Matrix - {best_name}",
    save_path="../outputs/figures/04_confusion_matrix.png")

# Classification report
print("\n" + get_classification_report(y_test, best_data["y_pred"]))

Best model: XGBoost

              precision    recall  f1-score   support

Not Canceled       0.88      0.92      0.90     12644
    Canceled       0.76      0.68      0.72      4750

    accuracy                           0.85     17394
   macro avg       0.82      0.80      0.81     17394
weighted avg       0.85      0.85      0.85     17394



## 5. ROC & PR Curves (tất cả models)

In [6]:
probas = {}
for name, data in trained_models.items():
    if data['y_proba'] is not None:
        probas[name] = data['y_proba']

plot_multi_roc_pr(y_test, probas, save_path="../outputs/figures/04_roc_pr_curves.png")

## 6. Feature Importance

In [7]:
best_model = trained_models[best_name]['model']
fi = get_feature_importance(best_model, X_train.columns.tolist(), top_n=20)
plot_feature_importance(fi, title=f"Feature Importance - {best_name}",
    save_path="../outputs/figures/04_feature_importance.png")
save_results(fi, "../outputs/tables/feature_importance")

[Report] Saved CSV: ..\outputs\tables\feature_importance.csv
[Report] Saved JSON: ..\outputs\tables\feature_importance.json


## 7. Lưu best model

In [8]:
save_model(best_model, "../outputs/models/best_model.joblib")
print(f"Best model ({best_name}) saved!")

[Supervised] Model saved to ../outputs/models/best_model.joblib
Best model (XGBoost) saved!


## Kết luận Classification### Kết quả so sánh 4 models| Model | Accuracy | F1 | PR-AUC | ROC-AUC | Train Time ||---|---|---|---|---|---|| **XGBoost** 🏆 | 85.3% | **0.716** | **0.809** | **0.917** | 0.71s || RandomForest | 77.2% | 0.673 | 0.740 | 0.885 | 1.46s || LogisticRegression | 72.7% | 0.604 | 0.620 | 0.812 | 8.68s || DecisionTree | 70.3% | 0.614 | 0.524 | 0.805 | 0.27s |### Nhận xét1. **XGBoost** cho kết quả tốt nhất trên cả PR-AUC (0.809) và ROC-AUC (0.917), vượt mục tiêu 0.72. **RandomForest** đứng thứ 2, recall cao (0.859) nhưng precision thấp hơn XGBoost3. **LogisticRegression** (baseline) chậm nhất (8.68s) do optimize trên nhiều features one-hot encoded4. **DecisionTree** (baseline) nhanh nhất (0.27s) nhưng PR-AUC thấp nhất (0.524) → overfitting/underfitting5. **Leakage check**: đã loại bỏ `reservation_status` trước khi train → kết quả genuinely tốt### Top-5 Feature Importance (XGBoost)1. `room_mismatch` (16.6%) — phòng xếp khác phòng đặt → khách bất mãn → huỷ2. `required_car_parking_spaces` (11.4%) — proxy cho family booking, ít huỷ3. `market_segment_Online TA` (10.7%) — OTA booking hay huỷ nhất4. `is_local` (8.1%) — khách Portugal ít huỷ hơn international5. `deposit_type_Non Refund` (6.0%) — đặt cọc không hoàn → gần 100% bị huỷ (pattern đặc biệt)→ Tiếp theo: Notebook 04b - Semi-supervised Learning